# Industrial Defect Vision — Google Colab GPU

Notebook oficial para ejecutar el pipeline PyTorch del proyecto sobre GPU remota sin duplicar la lógica de entrenamiento.

**Separación de responsabilidades**

- Código y notebook: GitHub.
- Dataset, checkpoints, figuras y logs: Google Drive.
- Dataset durante training: `/content/ksdd2` para evitar I/O lento desde Drive.
- Training: los mismos scripts de `scripts/` y la misma lógica de `src/`.

Antes de empezar, seleccioná **Runtime > Change runtime type > GPU**.

Estructura persistente esperada:

```text
MyDrive/Perso/portfolio/industrial-defect-vision/
├── data/
│   └── ksdd2.tar
└── artifacts/
    ├── checkpoints/
    ├── figures/
    └── logs/
```

`/content` es efímero. Todo resultado importante debe terminar en Drive.


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

REPO_URL = "https://github.com/santiagoAvellaR/industrial-defect-vision.git"
BRANCH = "feat/m4-segmentation-threshold-sweep"
REPO_DIR = "/content/industrial-defect-vision"

DRIVE_ROOT = "/content/drive/MyDrive/Perso/portfolio/industrial-defect-vision"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/data"
DRIVE_CHECKPOINT_DIR = f"{DRIVE_ROOT}/artifacts/checkpoints"
DRIVE_FIGURE_DIR = f"{DRIVE_ROOT}/artifacts/figures"
DRIVE_LOG_DIR = f"{DRIVE_ROOT}/artifacts/logs"

KSDD2_ARCHIVE = f"{DRIVE_DATA_DIR}/ksdd2.tar"
KSDD2_LOCAL = "/content/ksdd2"

print("Branch:", BRANCH)
print("Drive root:", DRIVE_ROOT)


In [ ]:
# ============================================================
# 2. MONTAR DRIVE Y CREAR DIRECTORIOS PERSISTENTES
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os

for path in [
    DRIVE_DATA_DIR,
    DRIVE_CHECKPOINT_DIR,
    DRIVE_FIGURE_DIR,
    DRIVE_LOG_DIR,
]:
    os.makedirs(path, exist_ok=True)

print("Drive montado y directorios listos.")


In [ ]:
# ============================================================
# 3. CLONAR UNA COPIA LIMPIA DEL REPO DESDE GITHUB
# ============================================================

import os
import shutil
import subprocess


def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

run(["git", "clone", REPO_URL, REPO_DIR])
run(["git", "switch", BRANCH], cwd=REPO_DIR)

os.chdir(REPO_DIR)
print("Repo:", os.getcwd())


In [ ]:
# ============================================================
# 4. INSTALAR EL PROYECTO
# ============================================================

import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    cwd=REPO_DIR,
    check=True,
)

print("Proyecto instalado.")


In [ ]:
# ============================================================
# 5. HELPERS DE LOGGING
# ============================================================

import os
import subprocess


def run_and_log(cmd, log_path, cwd=None):
    """Muestra stdout/stderr en Colab y guarda exactamente el mismo stream en Drive."""
    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    print("$", " ".join(map(str, cmd)))
    print("log:", log_path)

    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

    return return_code


In [ ]:
# ============================================================
# 6. VERIFICAR GPU / CUDA
# ============================================================

import subprocess
import torch

subprocess.run(["nvidia-smi"], check=False)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Activá GPU en Runtime > Change runtime type > GPU")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM [GB]:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
)


## Staging del dataset

No entrenamos leyendo miles de PNG directamente desde Drive.

```text
Google Drive
    ↓ copiar un único ksdd2.tar
/content
    ↓ extraer
/content/ksdd2
    ↓
DataLoader → GPU
```


In [ ]:
# ============================================================
# 7. COPIAR Y EXTRAER KSDD2 A /content
# ============================================================

import os
import shutil
import subprocess

if not os.path.exists(KSDD2_ARCHIVE):
    raise FileNotFoundError(
        f"No encontré {KSDD2_ARCHIVE}. "
        "Subí ksdd2.tar a la carpeta data/ del proyecto en Drive."
    )

local_tar = "/content/ksdd2.tar"

if os.path.exists(KSDD2_LOCAL):
    shutil.rmtree(KSDD2_LOCAL)
if os.path.exists(local_tar):
    os.remove(local_tar)

print("Copiando archive desde Drive...")
shutil.copy2(KSDD2_ARCHIVE, local_tar)

print("Extrayendo...")
subprocess.run(["tar", "-xf", local_tar, "-C", "/content"], check=True)

if not os.path.isdir(KSDD2_LOCAL):
    raise RuntimeError("El tar debe contener la carpeta ksdd2/")

file_count = 0
png_count = 0
for root, _, files in os.walk(KSDD2_LOCAL):
    file_count += len(files)
    png_count += sum(name.lower().endswith(".png") for name in files)

print("Archivos:", file_count)
print("PNG:", png_count)
print("Dataset local:", KSDD2_LOCAL)


In [ ]:
# ============================================================
# 8. TESTS
# ============================================================

import sys

TEST_LOG = f"{DRIVE_LOG_DIR}/pytest.log"
run_and_log(
    [sys.executable, "-m", "pytest"],
    TEST_LOG,
    cwd=REPO_DIR,
)


## Smoke test GPU

Run descartable de 1 epoch para validar CUDA, DataLoader, backward, validation y checkpointing. No cuenta como experimento ML.


In [ ]:
# ============================================================
# 9. SMOKE TEST GPU — 1 EPOCH
# ============================================================

import sys

SMOKE_LOG = f"{DRIVE_LOG_DIR}/smoke_test.log"

smoke_cmd = [
    sys.executable,
    "scripts/train_simple_unet.py",
    KSDD2_LOCAL,
    "--epochs", "1",
    "--batch-size", "4",
    "--num-workers", "2",
    "--pin-memory",
    "--learning-rate", "3e-4",
    "--threshold", "0.5",
    "--dice-positive-only",
    "--checkpoint", "/content/smoke_best_loss.pt",
    "--dice-checkpoint", "/content/smoke_best_dice.pt",
]

run_and_log(smoke_cmd, SMOKE_LOG, cwd=REPO_DIR)


## Benchmark de infraestructura

Mantenemos `batch_size=4` para no mezclar esta optimización con una variable que puede alterar la dinámica de entrenamiento.

Los benchmarks de `num_workers` y `pin_memory` **no son experimentos ML**.


In [ ]:
# ============================================================
# 10. BENCHMARK DE num_workers
# ============================================================

import json
import sys
import time


def bench(workers, pin=False):
    cmd = [
        sys.executable,
        "scripts/train_simple_unet.py",
        KSDD2_LOCAL,
        "--epochs", "1",
        "--batch-size", "4",
        "--num-workers", str(workers),
        "--learning-rate", "3e-4",
        "--threshold", "0.5",
        "--dice-positive-only",
        "--checkpoint", f"/content/bench_w{workers}_loss.pt",
        "--dice-checkpoint", f"/content/bench_w{workers}_dice.pt",
    ]
    if pin:
        cmd.append("--pin-memory")

    label = f"w{workers}_pin{int(pin)}"
    log_path = f"{DRIVE_LOG_DIR}/benchmark_{label}.log"

    start = time.perf_counter()
    run_and_log(cmd, log_path, cwd=REPO_DIR)
    return time.perf_counter() - start


results = {}
for workers in [0, 2, 4]:
    print(f"\n=== workers={workers}, pin_memory=False ===")
    results[f"w{workers}_pin0"] = bench(workers, pin=False)

print("\nRESULTADOS")
for name, seconds in results.items():
    print(f"{name}: {seconds:.2f} s")

with open(f"{DRIVE_LOG_DIR}/benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)


In [ ]:
# ============================================================
# 11. BENCHMARK DE pin_memory
# ============================================================

import json

BEST_WORKERS = 2  # editar si el benchmark anterior indica otro valor

print(f"workers={BEST_WORKERS}, pin_memory=True")
pin_time = bench(BEST_WORKERS, pin=True)
results[f"w{BEST_WORKERS}_pin1"] = pin_time

with open(f"{DRIVE_LOG_DIR}/benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Tiempo: {pin_time:.2f} s")


## Training real

Antes de ejecutar esta sección, asigná un nombre real como `EXP-014-...`.

El notebook bloquea explícitamente `EXP-XXX` para evitar sobrescribir resultados con un nombre ambiguo.


In [ ]:
# ============================================================
# 12. CONFIGURACIÓN DEL EXPERIMENTO REAL
# ============================================================

EXPERIMENT_NAME = "EXP-XXX"  # CAMBIAR antes de ejecutar training real

EPOCHS = 20
BATCH_SIZE = 4
LEARNING_RATE = 3e-4
THRESHOLD = 0.5
NUM_WORKERS = 2
PIN_MEMORY = True

if not EXPERIMENT_NAME or EXPERIMENT_NAME == "EXP-XXX":
    raise ValueError(
        "Definí EXPERIMENT_NAME antes de ejecutar un training real. "
        "Ejemplo: EXP-014-pos-weight-bce."
    )

BEST_LOSS_CHECKPOINT = f"{DRIVE_CHECKPOINT_DIR}/{EXPERIMENT_NAME}_best_loss.pt"
BEST_DICE_CHECKPOINT = f"{DRIVE_CHECKPOINT_DIR}/{EXPERIMENT_NAME}_best_dice.pt"
TRAIN_LOG = f"{DRIVE_LOG_DIR}/{EXPERIMENT_NAME}_training.log"
SWEEP_LOG = f"{DRIVE_LOG_DIR}/{EXPERIMENT_NAME}_threshold_sweep.log"

print("Best-loss checkpoint:", BEST_LOSS_CHECKPOINT)
print("Best-Dice checkpoint:", BEST_DICE_CHECKPOINT)
print("Training log:", TRAIN_LOG)


In [ ]:
# ============================================================
# 13. EJECUTAR TRAINING REAL + LOG PERSISTENTE
# ============================================================

import sys

train_cmd = [
    sys.executable,
    "scripts/train_simple_unet.py",
    KSDD2_LOCAL,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--learning-rate", str(LEARNING_RATE),
    "--threshold", str(THRESHOLD),
    "--dice-positive-only",
    "--checkpoint", BEST_LOSS_CHECKPOINT,
    "--dice-checkpoint", BEST_DICE_CHECKPOINT,
]

if PIN_MEMORY:
    train_cmd.append("--pin-memory")

run_and_log(train_cmd, TRAIN_LOG, cwd=REPO_DIR)


In [ ]:
# ============================================================
# 14. RESUMEN DEL BEST-DICE CHECKPOINT
# ============================================================

import json
import torch

checkpoint = torch.load(
    BEST_DICE_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

summary = {
    "experiment": EXPERIMENT_NAME,
    "epoch": checkpoint["epoch"],
    "val_loss": float(checkpoint["val_loss"]),
    "val_metrics": checkpoint["val_metrics"],
    "device_type": checkpoint.get("device_type"),
    "device_name": checkpoint.get("device_name"),
    "batch_size": checkpoint.get("batch_size"),
    "learning_rate": checkpoint.get("learning_rate"),
    "num_workers": checkpoint.get("num_workers"),
    "pin_memory": checkpoint.get("pin_memory"),
}

print(json.dumps(summary, indent=2))

summary_path = f"{DRIVE_LOG_DIR}/{EXPERIMENT_NAME}_best_checkpoint_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Resumen guardado en:", summary_path)


In [ ]:
# ============================================================
# 15. THRESHOLD SWEEP + LOG PERSISTENTE
# ============================================================

import sys

sweep_figure = f"{DRIVE_FIGURE_DIR}/{EXPERIMENT_NAME}_threshold_sweep.png"

sweep_cmd = [
    sys.executable,
    "scripts/evaluate_segmentation_thresholds.py",
    KSDD2_LOCAL,
    BEST_DICE_CHECKPOINT,
    "--figure", sweep_figure,
]

run_and_log(sweep_cmd, SWEEP_LOG, cwd=REPO_DIR)

print("Figura:", sweep_figure)
print("Log:", SWEEP_LOG)


## Antes de cerrar

Persistente en Drive:

```text
artifacts/
├── checkpoints/
├── figures/
└── logs/
```

Efímero en Colab:

```text
/content/industrial-defect-vision
/content/ksdd2
/content/ksdd2.tar
```

Para una nueva sesión: montar Drive → clonar GitHub → instalar → verificar CUDA → stagear KSDD2 → ejecutar.

El notebook oficial debe abrirse desde GitHub; no mantengas una segunda versión manual como fuente de verdad en Drive.
